# 🔬 LLM Evaluation / LLMOps — Days 6–10 Code-Along

**Sprint:** 14-Day AI Engineer Skills Sprint  
**Topic:** LLM Evaluation, RAGAS, LangSmith, Guardrails, CI/CD for AI  
**Pace:** ~2 hrs/day | Style: short concept read → heavy hands-on  

---

## How to use this notebook

- Each day has a **Concepts** section — read it even on the laziest day. It has the key mental models.  
- Each day has **📖 Must-Read Docs** — links to exact documentation to skim before coding.  
- Each day has **🧪 Try It Yourself** cells — attempt these yourself first.  
- Each day has **✅ Reference Solution** cells — complete, working code to fall back on.  
- Day 8 and Day 10 have LinkedIn post reminders.  

---

## Environment

- Python 3.10+  
- OpenAI API key (or swap with your Gemini/Bedrock adapter from Day 1–5)  
- LangSmith account (free tier) — set up by Day 8  
- GitHub repo with Actions enabled — needed for Day 10  

In [ ]:
# ── SETUP: Install dependencies ──────────────────────────────────────────────
# Run this once before starting Day 6.

# !pip install -U ragas langchain-openai langchain-community qdrant-client \
#              guardrails-ai langsmith python-dotenv datasets

import os
from dotenv import load_dotenv

load_dotenv()  # expects OPENAI_API_KEY (and LANGCHAIN_API_KEY for Day 8) in .env

import langchain_openai, ragas
print(f"langchain-openai : {langchain_openai.__version__}")
print(f"ragas            : {ragas.__version__}")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("LANGCHAIN_API_KEY set:", bool(os.getenv("LANGCHAIN_API_KEY")))

---
# 📅 Day 6 — What Does "Evaluating an LLM" Actually Mean?

**Time budget:** ~30 min concepts + 90 min hands-on

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| RAGAS Overview | What RAGAS measures and why | https://docs.ragas.io/en/latest/concepts/metrics/overview/ |
| RAGAS Metrics | Faithfulness, Answer Relevancy, Context Precision/Recall | https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/ |
| RAGAS Quickstart | Your first RAGAS evaluation | https://docs.ragas.io/en/latest/getstarted/rag_eval/ |

**Minimum read on a lazy day:** Read the RAGAS Overview page. Understand what Faithfulness means vs Answer Relevancy — they measure different failure modes.

---

## 🧠 Core Concepts

### Why vibes-based testing fails

"It sounds right" is not a test. In production:
- A prompt change can silently degrade quality — no error, just worse answers
- Retrieval can regress — you add a new doc, old docs get pushed out
- Hallucinations scale — 1 in 20 answers is hallucinated, you won't catch it manually

You need **reproducible metric scores** you can track over time.

### Offline vs online vs human eval

```
Offline eval   → run on a fixed test dataset, no real users, fast and cheap
Online eval    → run on real production traffic, sampled and scored async
Human eval     → gold standard but slow and expensive — use to calibrate, not scale
```

Start with offline eval. Add online eval once you're in prod.

### RAGAS — the 4 core metrics

```
Faithfulness         → is the answer grounded in the retrieved context?
                       Low score = LLM is hallucinating beyond the source docs

Answer Relevancy     → does the answer actually address the question asked?
                       Low score = answer is off-topic or too vague

Context Precision    → of the retrieved chunks, how many are relevant?
                       Low score = retrieval is noisy (lots of irrelevant chunks)

Context Recall       → did retrieval find all chunks needed to answer correctly?
                       Low score = retrieval is missing important chunks
```

### How RAGAS works

RAGAS uses an **LLM-as-judge** internally — it sends each metric prompt to an LLM (default: GPT-4o) to score. This means:
- Costs money (a few cents per evaluation)
- Quality depends on judge model quality
- You can swap the judge to your preferred model

### Input format

RAGAS expects each row in your test dataset to have:
```python
{
    "user_input": "question here",
    "response": "LLM's answer here",
    "retrieved_contexts": ["chunk1", "chunk2"],  # your RAG pipeline's retrieved docs
    "reference": "ground truth answer here",      # needed for some metrics
}
```

### 🧪 Try It Yourself — 6.1: Build a minimal RAG test dataset

Create a Python list of 5 sample QA pairs in the RAGAS input format.
Use a topic you know — these can be hand-crafted (no real RAG pipeline needed yet).

Each item: `user_input`, `response`, `retrieved_contexts` (list of 2 strings), `reference`.

Then wrap it in a `datasets.Dataset` using `datasets.Dataset.from_list(your_list)`.

In [ ]:
# 🧪 YOUR CODE HERE — Day 6.1: Build a minimal RAG test dataset
# from datasets import Dataset
# Create 5 QA pairs, wrap in Dataset

In [ ]:
# ✅ REFERENCE SOLUTION — Day 6.1: Minimal RAGAS test dataset
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. 'retrieved_contexts' is a LIST of strings — each string is one retrieved chunk.
# 2. 'reference' is the ground-truth answer — used only for Context Recall.
#    You can omit it if you only compute Faithfulness + Answer Relevancy.
# 3. Hand-crafted datasets are FINE for learning. In production, you generate
#    them from real user queries + your pipeline's actual retrieved chunks.

from datasets import Dataset

raw_data = [
    {
        "user_input": "What is LangGraph used for?",
        "response": "LangGraph is used to build stateful, multi-actor applications with LLMs, especially for agentic workflows that require cycles and branching.",
        "retrieved_contexts": [
            "LangGraph is a library for building stateful, multi-actor applications with LLMs.",
            "It extends LangChain by enabling cyclic graphs for agentic workflows."
        ],
        "reference": "LangGraph is used to build stateful, multi-actor agentic applications with LLMs."
    },
    {
        "user_input": "What is the difference between invoke() and stream() in LangGraph?",
        "response": "invoke() runs the full graph and returns the final state. stream() yields the state update after each node finishes, which is useful for real-time UIs.",
        "retrieved_contexts": [
            "invoke() blocks until the full graph finishes and returns the final state.",
            "stream() yields intermediate state updates after each node runs."
        ],
        "reference": "invoke() returns final state; stream() yields per-node state updates."
    },
    {
        "user_input": "What is a conditional edge in LangGraph?",
        "response": "A conditional edge uses a router function to decide which node to visit next based on the current state.",
        "retrieved_contexts": [
            "add_conditional_edges() takes a router function that reads state and returns the next node name.",
            "Conditional edges allow LLM-driven routing decisions within a graph."
        ],
        "reference": "A conditional edge calls a router function to dynamically determine the next node."
    },
    {
        "user_input": "What checkpointers does LangGraph support?",
        "response": "LangGraph supports MemorySaver for in-memory persistence, SqliteSaver for SQLite, and a Postgres checkpointer for production use.",
        "retrieved_contexts": [
            "MemorySaver stores state in-process and is lost when the process ends.",
            "SqliteSaver persists state to a SQLite file and survives restarts."
        ],
        "reference": "LangGraph supports MemorySaver, SqliteSaver, and a Postgres checkpointer."
    },
    {
        "user_input": "How do you deploy a LangGraph agent to production?",
        "response": "You can wrap a LangGraph graph in a FastAPI app, containerise it with Docker, and deploy to AWS EC2 or ECS via ECR.",
        "retrieved_contexts": [
            "Wrap your graph in a FastAPI POST /invoke endpoint for synchronous calls.",
            "Docker + ECR + EC2 is a common deployment path for LangGraph agents."
        ],
        "reference": "Wrap in FastAPI, containerise with Docker, deploy to AWS EC2 or ECS."
    },
]

eval_dataset = Dataset.from_list(raw_data)
print(f"Dataset size: {len(eval_dataset)} rows")
print(f"Columns: {eval_dataset.column_names}")
print("\nFirst row user_input:", eval_dataset[0]["user_input"])

### 🧪 Try It Yourself — 6.2: Run RAGAS evaluation

Using the dataset from 6.1:
1. Import `evaluate` from `ragas` and import the metrics: `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`
2. Run `evaluate(dataset, metrics=[...])` — this will call GPT-4o as judge (uses your API key)
3. Print the results as a pandas DataFrame
4. Now deliberately corrupt one `response` with a hallucination and re-run — see Faithfulness drop

In [ ]:
# 🧪 YOUR CODE HERE — Day 6.2: Run RAGAS evaluation
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# results = evaluate(eval_dataset, metrics=[...])
# print(results.to_pandas())

In [ ]:
# ✅ REFERENCE SOLUTION — Day 6.2: Run RAGAS and observe score changes
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. evaluate() sends each row to an LLM judge — expect a few seconds per row.
# 2. Faithfulness checks: is each claim in the response SUPPORTED by the context?
#    Hallucinated response → Faithfulness drops toward 0.
# 3. Context Precision checks retrieved chunks — if you add irrelevant chunks
#    to 'retrieved_contexts', this score drops. Try it!
# 4. You can customise the judge LLM: pass llm=ChatOpenAI(model="gpt-4o") to evaluate()

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_openai import ChatOpenAI

# Run evaluation on the clean dataset
print("=== Evaluating clean dataset ===")
results_clean = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)
df_clean = results_clean.to_pandas()
print(df_clean[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]])
print("\nMean scores:")
print(df_clean[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].mean())

# Now corrupt one answer — insert a hallucination
hallucinated_data = raw_data.copy()
hallucinated_data[0]["response"] = (
    "LangGraph was invented in 2019 by researchers at MIT and is "
    "primarily used for robotics control systems."
)
hallucinated_dataset = Dataset.from_list(hallucinated_data)

print("\n=== Evaluating hallucinated dataset (row 0 corrupted) ===")
results_noisy = evaluate(
    hallucinated_dataset,
    metrics=[faithfulness, answer_relevancy],
)
df_noisy = results_noisy.to_pandas()
print(df_noisy[["user_input", "faithfulness", "answer_relevancy"]].head(2))
print("\n→ Notice Row 0 Faithfulness drop — hallucination detected!")

### ✅ Day 6 Checkpoint

You should now be able to explain, without looking anything up:
- [ ] What Faithfulness measures vs Answer Relevancy — they are NOT the same thing  
- [ ] Why Context Precision and Context Recall matter separately (precision = quality, recall = coverage)  
- [ ] Why a hallucination lowers Faithfulness but may NOT lower Answer Relevancy  
- [ ] What RAGAS uses internally to score (LLM-as-judge)  

**If you can explain a score of 0.62 Context Precision to a non-technical stakeholder — Day 6 is done.**

---

# 📅 Day 7 — Building a Custom Eval Framework

**Time budget:** ~20 min concepts + 100 min hands-on

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| RAGAS Custom Metrics | How to add domain-specific metrics | https://docs.ragas.io/en/latest/howtos/customisations/custom_metrics/ |
| LangChain Structured Output | `with_structured_output` for JSON scores | https://python.langchain.com/docs/how_to/structured_output/ |
| Pydantic V2 | Schema validation for eval outputs | https://docs.pydantic.dev/latest/concepts/models/ |

**Minimum read on a lazy day:** Read the structured output how-to. The pattern of using `with_structured_output(schema)` to force JSON from an LLM is fundamental — you'll use it constantly.

---

## 🧠 Core Concepts

### When RAGAS is not enough

RAGAS measures generic RAG quality. It does NOT measure:
- Domain-specific correctness (e.g. "is this ANZSIC code accurate?")
- Tone and style adherence (e.g. "is the response professional enough?")
- Procedural compliance (e.g. "did the agent follow the required steps?")

For these, you write a **custom LLM-as-judge evaluator**.

### LLM-as-a-judge pattern

```
INPUT:  question, llm_answer, ground_truth
JUDGE:  strong LLM (GPT-4o or Claude) with a rubric prompt
OUTPUT: structured JSON scores { correctness: 0-5, conciseness: 0-5, tone: 0-5 }
```

Key to reliability: **force structured output** using `with_structured_output(Pydantic schema)`. Never ask the judge to return JSON in plain text — it will sometimes deviate.

### Structuring your eval dataset

```python
# Minimum fields for a custom eval row:
{
    "id": "test_001",          # unique ID for tracking regressions
    "question": "...",
    "ground_truth": "...",     # what the correct answer IS
    "agent_response": "...",   # what your agent actually said
    "metadata": {}             # optional: version, date, tags
}
```

Version your test dataset in Git. Treat it like production code.

### 🧪 Try It Yourself — 7.1: Write a custom LLM-as-judge evaluator

1. Define a Pydantic schema `EvalScore` with fields: `correctness: int` (0–5), `conciseness: int` (0–5), `tone: int` (0–5), `reasoning: str`
2. Write a function `evaluate_response(question, answer, ground_truth)` that:
   - Builds a judge prompt with a rubric
   - Calls GPT-4o (or your LLM) with `with_structured_output(EvalScore)`
   - Returns the `EvalScore` object
3. Test on 2 answers — one good, one poor

In [ ]:
# 🧪 YOUR CODE HERE — Day 7.1: Custom LLM-as-judge evaluator
# from pydantic import BaseModel
# from langchain_openai import ChatOpenAI
# class EvalScore(BaseModel): ...
# def evaluate_response(question, answer, ground_truth): ...

In [ ]:
# ✅ REFERENCE SOLUTION — Day 7.1: Custom LLM-as-judge with structured output
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. with_structured_output(schema) forces the LLM to return valid JSON matching
#    the Pydantic schema — no manual JSON parsing, no format failures.
# 2. The rubric in the system prompt is everything. Vague rubric → inconsistent scores.
#    Be specific: define what a 5 vs a 3 vs a 1 looks like for each dimension.
# 3. 'reasoning' field forces the judge to explain its scores — this makes
#    the eval output auditable and helps you debug bad scores.

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

class EvalScore(BaseModel):
    correctness: int = Field(description="0-5: how factually correct is the answer vs ground truth")
    conciseness: int = Field(description="0-5: does the answer avoid unnecessary verbosity")
    tone:        int = Field(description="0-5: is the tone professional and appropriate")
    reasoning:   str = Field(description="Brief explanation of the scores given")


JUDGE_SYSTEM = """
You are an expert evaluator for AI assistant responses.
Score the response on three dimensions, each from 0 to 5:

CORRECTNESS (0-5):
  5 = fully correct, matches ground truth exactly
  3 = partially correct, missing key details
  1 = mostly wrong
  0 = completely wrong or hallucinated

CONCISENESS (0-5):
  5 = answers directly, no fluff
  3 = answer is correct but verbose
  1 = answer is buried in unnecessary text

TONE (0-5):
  5 = professional, clear, appropriate
  3 = acceptable but informal
  1 = unprofessional or confusing
"""

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_judge = judge_llm.with_structured_output(EvalScore)


def evaluate_response(question: str, answer: str, ground_truth: str) -> EvalScore:
    """Score a single LLM answer using the LLM-as-judge pattern."""
    user_prompt = f"""
QUESTION: {question}
GROUND TRUTH: {ground_truth}
RESPONSE TO EVALUATE: {answer}

Score the response on correctness, conciseness, and tone.
"""
    return structured_judge.invoke([
        SystemMessage(content=JUDGE_SYSTEM),
        HumanMessage(content=user_prompt),
    ])


# Test on two answers
q = "What is the capital of Australia?"
gt = "Canberra"

good_answer = "Canberra is the capital of Australia."
poor_answer = "Sydney is the capital of Australia and it is a very beautiful city with the famous Opera House."

print("=== Good answer ===")
score_good = evaluate_response(q, good_answer, gt)
print(score_good.model_dump_json(indent=2))

print("\n=== Poor answer (wrong + verbose) ===")
score_poor = evaluate_response(q, poor_answer, gt)
print(score_poor.model_dump_json(indent=2))

### 🧪 Try It Yourself — 7.2: Build an eval runner

1. Create a JSON file `eval_cases.json` with 5 test cases (question, ground_truth, agent_response)
2. Write a function `run_eval(filepath)` that:
   - Loads the JSON
   - Runs each case through `evaluate_response()`
   - Prints a summary table: id, correctness, conciseness, tone, avg_score
   - Flags any row where avg_score < 3.0 as `⚠️ FAIL`
3. Run it and observe which cases fail

In [ ]:
# 🧪 YOUR CODE HERE — Day 7.2: Eval runner from JSON file
# import json
# def run_eval(filepath: str): ...

In [ ]:
# ✅ REFERENCE SOLUTION — Day 7.2: Eval runner with summary report
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. Loading test cases from a JSON file (not hardcoded) is essential —
#    you can add new cases without touching eval code.
# 2. Version this JSON in Git. When a new agent version changes a score,
#    you will see it in git diff — that's regression tracking.
# 3. A threshold (avg < 3.0 = FAIL) is your CI gate trigger (Day 10 uses this).

import json
from pathlib import Path

# Create a sample eval_cases.json inline (in production: check this into Git)
EVAL_CASES = [
    {"id": "001", "question": "What is the capital of Australia?", "ground_truth": "Canberra",
     "agent_response": "Canberra is the capital of Australia."},
    {"id": "002", "question": "What year was Python created?", "ground_truth": "1991",
     "agent_response": "Python was created in 1991 by Guido van Rossum."},
    {"id": "003", "question": "What does RAG stand for?", "ground_truth": "Retrieval-Augmented Generation",
     "agent_response": "I believe RAG might stand for something related to data retrieval."},
    {"id": "004", "question": "What is asyncio used for in Python?", "ground_truth": "Writing concurrent I/O-bound code using async/await",
     "agent_response": "asyncio is a Python library for writing concurrent code using the async/await syntax, mainly for I/O-bound tasks."},
    {"id": "005", "question": "What is a LangGraph StateGraph?", "ground_truth": "A graph container that holds nodes, edges, and a shared state TypedDict",
     "agent_response": "It is a complicated thing in LangChain used for building stuff with LLMs and graphs and state."},
]

# Save to file
eval_path = Path("eval_cases.json")
eval_path.write_text(json.dumps(EVAL_CASES, indent=2))
print(f"Saved {len(EVAL_CASES)} test cases to {eval_path}")


def run_eval(filepath: str, threshold: float = 3.0) -> None:
    """Load test cases, evaluate each, print summary. Flag failures."""
    cases = json.loads(Path(filepath).read_text())
    
    print(f"\n{'─'*70}")
    print(f"{'ID':<6} {'Correct':>8} {'Concise':>8} {'Tone':>6} {'Avg':>6}  Status")
    print(f"{'─'*70}")
    
    failed = 0
    for case in cases:
        score = evaluate_response(
            question=case["question"],
            answer=case["agent_response"],
            ground_truth=case["ground_truth"],
        )
        avg = (score.correctness + score.conciseness + score.tone) / 3
        status = "✅ PASS" if avg >= threshold else "⚠️  FAIL"
        if avg < threshold:
            failed += 1
        print(f"{case['id']:<6} {score.correctness:>8} {score.conciseness:>8} {score.tone:>6} {avg:>6.2f}  {status}")
    
    print(f"{'─'*70}")
    print(f"Result: {len(cases) - failed}/{len(cases)} passed (threshold={threshold})")
    if failed > 0:
        print(f"→ {failed} case(s) below threshold — would BLOCK a CI merge.")


run_eval("eval_cases.json")

### ✅ Day 7 Checkpoint

- [ ] You can explain why `with_structured_output()` is better than asking the LLM to return JSON in the prompt  
- [ ] You know what makes a good judge rubric (specificity — define each score level explicitly)  
- [ ] You have a versioned eval dataset in a JSON file  
- [ ] Your eval runner prints a pass/fail summary you could pipe into CI  

---

# 📅 Day 8 — Tracing and Observability with LangSmith

**Time budget:** ~20 min concepts + 100 min hands-on  
**📝 Write LinkedIn Post #3 tonight** — see `02_linkedin_posts.md`

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| LangSmith Quickstart | Set up tracing in 2 env vars | https://docs.smith.langchain.com/observability/quickstart |
| Tracing Concepts | Runs, traces, spans | https://docs.smith.langchain.com/observability/concepts |
| Datasets in LangSmith | Creating datasets from traces | https://docs.smith.langchain.com/evaluation/concepts |

**Minimum read on a lazy day:** Read the Quickstart. All you need is two env vars — the tracing happens automatically.

---

## 🧠 Core Concepts

### What LangSmith traces

```
Every LLM call    → tokens in/out, latency, model used, cost estimate
Every LangGraph node → which node ran, inputs, outputs, duration
Every tool call   → tool name, args, result, latency
Errors            → exact stack trace, which node failed, what state was at that point
```

**Zero code changes required.** Set 2 env vars and it works:

```bash
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__...
LANGCHAIN_PROJECT=my-project-name   # optional but recommended
```

### Anatomy of a trace

```
Run (top-level)           ← your graph.invoke() call
  ├── Node: supervisor    ← each LangGraph node is a child span
  │     └── LLM call      ← the actual OpenAI/Gemini API call inside the node
  ├── Node: research
  │     └── LLM call
  └── Node: writer
        └── LLM call
```

### Metadata tagging — essential for production

```python
graph.invoke(
    state,
    config={"metadata": {"user_id": "u123", "session_id": "s456", "version": "v2.1"}}
)
```

Without metadata, your traces are a pile of anonymous calls. With it, you can filter by user, session, or version in the LangSmith UI.

### 🧪 Try It Yourself — 8.1: Enable LangSmith tracing

1. Sign up at https://smith.langchain.com (free)
2. Create an API key and add to your `.env`:
   ```
   LANGCHAIN_TRACING_V2=true
   LANGCHAIN_API_KEY=ls__your_key_here
   LANGCHAIN_PROJECT=14day-sprint
   ```
3. Run any LangGraph graph from Days 1–5 (or the minimal one below)
4. Open LangSmith UI → verify the trace appears
5. Click into the trace — find the LLM call, note the token counts and latency

In [ ]:
# 🧪 YOUR CODE HERE — Day 8.1: Run a traced graph invocation
# 1. Load .env (make sure LANGCHAIN_TRACING_V2=true is set)
# 2. Import and invoke any graph from Days 1-5
# 3. Open LangSmith and find the trace

In [ ]:
# ✅ REFERENCE SOLUTION — Day 8.1: Traced graph with metadata tagging
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. LangSmith tracing is purely env-var driven — no code changes to your graph.
#    This is the right design: observability is infrastructure, not app logic.
# 2. The `metadata` dict in config shows up as filterable fields in LangSmith UI.
#    Always tag version, user_id, session_id from day 1 of production.
# 3. run_name gives the trace a human-readable name in the UI — much better
#    than the default UUID.

import os
from dotenv import load_dotenv
load_dotenv()

# Verify tracing is enabled
tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2", "false").lower() == "true"
print(f"LangSmith tracing enabled: {tracing_enabled}")
print(f"Project: {os.getenv('LANGCHAIN_PROJECT', '(not set — traces go to default project)')}")

# Build a minimal 2-node graph (reuse Day 1 pattern)
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class TracedState(TypedDict):
    query: str
    response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_node(state: TracedState) -> dict:
    result = llm.invoke([HumanMessage(content=state["query"])])
    return {"response": result.content}

builder = StateGraph(TracedState)
builder.add_node("llm_node", llm_node)
builder.set_entry_point("llm_node")
builder.add_edge("llm_node", END)
traced_app = builder.compile()

# Invoke with metadata — this shows up as filterable fields in LangSmith
result = traced_app.invoke(
    {"query": "What are the top 3 benefits of LangSmith?", "response": ""},
    config={
        "run_name": "day8-tracing-demo",
        "metadata": {
            "user_id": "demo_user",
            "session_id": "session_001",
            "version": "v1.0",
            "sprint_day": 8,
        }
    }
)
print("Response:", result["response"])
print("\n→ Check https://smith.langchain.com — your trace should appear within seconds.")

### 🧪 Try It Yourself — 8.2: Explore traces + create a LangSmith dataset

1. Run 5–10 invocations of your traced graph with varied queries
2. In LangSmith UI: find the slowest run (sort by latency)
3. In LangSmith UI: find the most expensive run (sort by total tokens)
4. Select 3 good runs → "Add to Dataset" → create a dataset called `sprint-eval-v1`
5. Print the dataset URL

In [ ]:
# 🧪 YOUR CODE HERE — Day 8.2: Run 10 invocations with varied queries
# Add tags and metadata to each run for filtering
QUERIES = [
    "Explain the event loop in asyncio",
    "What is a LangGraph StateGraph?",
    "How does RAGAS evaluate faithfulness?",
    "What is the supervisor pattern in multi-agent systems?",
    "Explain Docker in one sentence",
]
# Run each query through traced_app with metadata tagging
# Then go to LangSmith UI and explore

In [ ]:
# ✅ REFERENCE SOLUTION — Day 8.2: Batch runs with metadata + LangSmith dataset
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. Tagging each run with 'tags' allows filtering in LangSmith by topic or category.
# 2. Creating a dataset from production traces is the gold standard workflow:
#    real user queries → evaluated → good examples → become your eval set.
# 3. The LangSmith client SDK gives you programmatic access to all your traces.

from langsmith import Client

QUERIES_WITH_TAGS = [
    ("Explain the event loop in asyncio", ["asyncio", "concepts"]),
    ("What is a LangGraph StateGraph?", ["langgraph", "concepts"]),
    ("How does RAGAS evaluate faithfulness?", ["eval", "ragas"]),
    ("What is the supervisor pattern in multi-agent systems?", ["langgraph", "multi-agent"]),
    ("Explain Docker in one sentence", ["deployment", "infra"]),
]

print("Running 5 traced invocations...")
for i, (query, tags) in enumerate(QUERIES_WITH_TAGS, 1):
    result = traced_app.invoke(
        {"query": query, "response": ""},
        config={
            "run_name": f"batch-run-{i:02d}",
            "tags": tags,
            "metadata": {"batch": "day8-demo", "query_index": i}
        }
    )
    print(f"  [{i}] {query[:40]}... → {result['response'][:60]}...")

print("\n→ All 5 runs traced. Go to LangSmith UI to:")
print("   1. Filter by tag 'langgraph' — see only LangGraph queries")
print("   2. Sort by latency — find your slowest run")
print("   3. Select 3 runs → Add to Dataset → name it 'sprint-eval-v1'")

# Optional: programmatically list your datasets if LANGCHAIN_API_KEY is set
if os.getenv("LANGCHAIN_API_KEY"):
    client = Client()
    datasets = list(client.list_datasets())
    if datasets:
        print(f"\nYour LangSmith datasets: {[d.name for d in datasets]}")
    else:
        print("\nNo datasets yet — create one in the UI after this run.")

### ✅ Day 8 Checkpoint

- [ ] LangSmith is tracing your agent automatically — you saw it in the UI  
- [ ] You can find the slowest node and the most token-intensive call  
- [ ] You know why metadata tagging is not optional in production  
- [ ] You have at least one dataset in LangSmith built from real traces  

**📝 Write LinkedIn Post #3 tonight** — open `02_linkedin_posts.md` → Post 3 (Day 8)

---

# 📅 Day 9 — Guardrails: Controlling LLM Inputs and Outputs

**Time budget:** ~30 min concepts + 90 min hands-on

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| Guardrails AI Getting Started | Install, validators, basic usage | https://www.guardrailsai.com/docs/getting_started/quickstart |
| Pydantic Validators | Custom validators for structured data | https://docs.pydantic.dev/latest/concepts/validators/ |
| NVIDIA NeMo Guardrails | Alternative framework, config-driven | https://github.com/NVIDIA/NeMo-Guardrails |

**Minimum read on a lazy day:** Read the Guardrails AI Quickstart. Understand the Guard → validator → LLM call → validated output flow.

---

## 🧠 Core Concepts

### What guardrails solve

```
Hallucination      → response contains facts not in context
Prompt injection   → user input hijacks the system prompt
Off-topic input    → user asks about something outside your domain
PII leakage        → response contains emails, phone numbers, names
Schema violation   → response isn't the JSON shape your code expects
```

### Where to place guardrails in a LangGraph agent

```
User input →  [INPUT GUARDRAIL node]  →  agent graph  →  [OUTPUT GUARDRAIL node]  →  response
                    ↓ blocks                                        ↓ blocks
               rejection message                              regeneration or block
```

Input guardrails are cheaper (no LLM call wasted). Output guardrails are the last line of defence.

### Three practical approaches

1. **Guardrails AI** — declarative validators, hub of pre-built rails, wraps any LLM  
2. **Pydantic + `with_structured_output`** — enforce output schema (lightest, no extra library)  
3. **Custom LangGraph node** — write your own validator as just another node  

In most real systems: Pydantic for schema enforcement + a custom input filter node + optional Guardrails AI for PII/topic rails.

### 🧪 Try It Yourself — 9.1: Schema guardrail with Pydantic

Build a node that forces the LLM to always output a structured classification:
1. Define a Pydantic `ClassificationResult` with: `category: str`, `confidence: float` (0–1), `reasoning: str`
2. Use `llm.with_structured_output(ClassificationResult)` as the LLM call
3. Add a validator: if `confidence < 0.5`, raise a `ValueError` with a clear message
4. Test with a clear query (high confidence) and an ambiguous one (should trigger the validator)

In [ ]:
# 🧪 YOUR CODE HERE — Day 9.1: Pydantic schema guardrail
# from pydantic import BaseModel, field_validator
# class ClassificationResult(BaseModel): ...

In [ ]:
# ✅ REFERENCE SOLUTION — Day 9.1: Schema + confidence guardrail
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. with_structured_output() is your first guardrail — it ensures the LLM
#    response always matches your schema. No JSON parsing errors in prod.
# 2. Pydantic field_validator runs on the parsed output — use it to enforce
#    business rules (confidence threshold, category whitelist, etc.).
# 3. Wrapping the LLM call in try/except and returning a fallback is the
#    production pattern — never let a guardrail crash your agent silently.

from pydantic import BaseModel, Field, field_validator
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

VALID_CATEGORIES = {"technical", "billing", "general", "off-topic"}

class ClassificationResult(BaseModel):
    category:   str   = Field(description=f"One of: {', '.join(sorted(VALID_CATEGORIES))}")
    confidence: float = Field(description="Confidence score 0.0 to 1.0")
    reasoning:  str   = Field(description="Why you chose this category")

    @field_validator("confidence")
    @classmethod
    def confidence_must_be_valid(cls, v: float) -> float:
        if not 0.0 <= v <= 1.0:
            raise ValueError(f"Confidence must be 0-1, got {v}")
        return v

    @field_validator("category")
    @classmethod
    def category_must_be_known(cls, v: str) -> str:
        if v.lower() not in VALID_CATEGORIES:
            raise ValueError(f"Unknown category '{v}'. Must be one of {VALID_CATEGORIES}")
        return v.lower()


CLASSIFIER_SYSTEM = (
    "Classify customer queries into: technical, billing, general, or off-topic. "
    "Be honest about confidence — return low confidence for ambiguous queries."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
classifier = llm.with_structured_output(ClassificationResult)


def classify_query(query: str) -> ClassificationResult | None:
    """Classify a query. Returns None if confidence < 0.5 (guardrail fires)."""
    try:
        result = classifier.invoke([
            SystemMessage(content=CLASSIFIER_SYSTEM),
            HumanMessage(content=query),
        ])
        if result.confidence < 0.5:
            print(f"  ⚠️  Guardrail fired — confidence {result.confidence:.2f} < 0.5 for: {query!r}")
            return None
        return result
    except Exception as e:
        print(f"  ❌ Validation error: {e}")
        return None


test_queries = [
    "My API key is not working, I keep getting 401 errors",  # clear: technical
    "How much will I be charged next month?",                 # clear: billing
    "Can you recommend a good restaurant nearby?",            # off-topic
    "Is it maybe possible the thing could be whatever?",      # ambiguous → guardrail
]

for q in test_queries:
    print(f"\nQuery: {q!r}")
    result = classify_query(q)
    if result:
        print(f"  → category={result.category}, confidence={result.confidence:.2f}")
        print(f"     reasoning: {result.reasoning[:80]}...")

### 🧪 Try It Yourself — 9.2: Input + output guardrail nodes in LangGraph

Integrate guardrails as **first and last nodes** in a simple LangGraph agent:
1. `input_guardrail_node`: checks if query is on-topic (use classify_query). If off-topic → set `blocked=True` in state, add rejection message  
2. `llm_node`: only runs if `blocked=False`  
3. `output_guardrail_node`: checks response length < 500 chars and doesn't contain words like "confidential"  
4. Wire with conditional edge: after input_guardrail, if `blocked=True` → END, else → llm_node

In [ ]:
# 🧪 YOUR CODE HERE — Day 9.2: Guardrailed LangGraph agent
# Build a 3-node graph: input_guardrail → llm_node → output_guardrail
# Conditional edge after input: blocked → END, not blocked → llm_node

In [ ]:
# ✅ REFERENCE SOLUTION — Day 9.2: Guardrailed LangGraph agent
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. Guardrails as nodes (not middleware) gives you full visibility in LangSmith —
#    you can trace which queries were blocked and why.
# 2. The `blocked` field in state is the routing signal — a clean pattern for any
#    conditional early-exit in a graph.
# 3. Output guardrails should never throw — they should return a sanitised response
#    or a generic fallback. Throwing crashes the whole run.

from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

BLOCKED_WORDS = {"confidential", "classified", "internal only", "do not share"}

class GuardrailedState(TypedDict):
    query:    str
    response: str
    blocked:  bool
    block_reason: Optional[str]


def input_guardrail_node(state: GuardrailedState) -> dict:
    """Block off-topic queries before they reach the LLM."""
    result = classify_query(state["query"])
    if result is None or result.category == "off-topic":
        print(f"  🛡️  INPUT GUARDRAIL blocked: {state['query']!r}")
        return {
            "blocked": True,
            "response": "I can only help with technical or billing questions. Please rephrase.",
            "block_reason": "off-topic or low confidence",
        }
    print(f"  ✅ Input passed — category: {result.category} ({result.confidence:.2f})")
    return {"blocked": False, "block_reason": None}


def llm_node_guarded(state: GuardrailedState) -> dict:
    """Main LLM response — only runs if input passed."""
    response = llm.invoke([HumanMessage(content=state["query"])])
    return {"response": response.content}


def output_guardrail_node(state: GuardrailedState) -> dict:
    """Sanitise the LLM response before returning to the user."""
    resp = state["response"]
    # Rule 1: length check
    if len(resp) > 500:
        resp = resp[:497] + "..."
        print("  🛡️  OUTPUT GUARDRAIL: response truncated to 500 chars")
    # Rule 2: blocked word check
    for word in BLOCKED_WORDS:
        if word.lower() in resp.lower():
            print(f"  🛡️  OUTPUT GUARDRAIL: blocked word '{word}' found — replacing response")
            return {"response": "[Response withheld — contains restricted content.]", "blocked": True}
    return {"response": resp}


def input_router(state: GuardrailedState) -> str:
    return END if state["blocked"] else "llm_node"


builder = StateGraph(GuardrailedState)
builder.add_node("input_guardrail", input_guardrail_node)
builder.add_node("llm_node", llm_node_guarded)
builder.add_node("output_guardrail", output_guardrail_node)
builder.set_entry_point("input_guardrail")
builder.add_conditional_edges("input_guardrail", input_router)
builder.add_edge("llm_node", "output_guardrail")
builder.add_edge("output_guardrail", END)
guardrailed_app = builder.compile()

# Test
test_inputs = [
    "My API keeps returning 401 errors",          # should pass
    "Tell me a joke about programmers",           # should be blocked (off-topic)
    "What is my billing cycle?",                  # should pass
]

for q in test_inputs:
    print(f"\n{'='*50}\nQuery: {q}")
    out = guardrailed_app.invoke(
        {"query": q, "response": "", "blocked": False, "block_reason": None}
    )
    print(f"Response: {out['response'][:100]}")
    print(f"Blocked: {out['blocked']}")

### ✅ Day 9 Checkpoint

- [ ] You can explain where to put guardrails in an agent graph and why (input = early exit, output = last defence)  
- [ ] You've used `with_structured_output` + Pydantic validators as a guardrail  
- [ ] You have a working input + output guardrail node pair in LangGraph  
- [ ] You know the difference between "block and reject" vs "sanitise and continue"  

---

# 📅 Day 10 — CI/CD for AI: Automated Eval on Every Change

**Time budget:** ~20 min concepts + 100 min hands-on  
**📝 Write LinkedIn Post #4 tonight** — see `02_linkedin_posts.md`

---

## 📖 Must-Read Docs

| Doc | What you'll learn | URL |
|---|---|---|
| GitHub Actions Quickstart | Workflow YAML syntax | https://docs.github.com/en/actions/writing-workflows/quickstart |
| LangSmith Pytest Integration | Running evals in CI with pytest | https://docs.smith.langchain.com/evaluation/how_to_guides/pytest |
| GitHub Actions: Secrets | Storing API keys securely | https://docs.github.com/en/actions/security-guides/using-secrets-in-github-actions |

**Minimum read on a lazy day:** Read the GitHub Actions Quickstart — understand triggers, jobs, and steps. The eval pipeline just adds an extra step.

---

## 🧠 Core Concepts

### What LLMOps CI/CD looks like

```
Developer makes a change (prompt, model, retrieval config)
       ↓
Opens a Pull Request on GitHub
       ↓
GitHub Actions triggers → runs eval pipeline:
   1. pip install deps
   2. run eval script against test dataset
   3. collect metric scores
   4. compare to thresholds:
      Faithfulness ≥ 0.85?  Answer Relevancy ≥ 0.80?  Avg correctness ≥ 3.0?
       ↓ pass                          ↓ fail
   PR gets green check          PR is blocked — "eval failed"
```

### Why this matters

Without this: a prompt change silently ships to production, 3 days later someone notices answers got worse, nobody knows which commit did it.  
With this: the commit that degraded quality is blocked at merge time. The team sees exactly which metric dropped and by how much.

### Regression testing for LLMs

Run both old and new agent versions on the same fixed dataset. If new_score < old_score - tolerance → fail.

```python
THRESHOLDS = {
    "faithfulness": 0.85,
    "answer_relevancy": 0.80,
    "avg_correctness": 3.0,   # from your custom evaluator
}
```

### 🧪 Try It Yourself — 10.1: Write a CI-ready eval script

Write a standalone `eval_ci.py` script that:
1. Loads `eval_cases.json`
2. Runs each case through your custom evaluator from Day 7
3. Calculates mean scores
4. Compares each to `THRESHOLDS`
5. Exits with `sys.exit(1)` if any threshold is breached — critical for CI
6. Prints a clean summary table

The exit code 1 is what tells GitHub Actions to fail the job.

In [ ]:
# 🧪 YOUR CODE HERE — Day 10.1: CI-ready eval script
# Write the script that will be called from GitHub Actions
# Key: sys.exit(1) if any threshold breached, sys.exit(0) if all pass

In [ ]:
# ✅ REFERENCE SOLUTION — Day 10.1: CI-ready eval script (save as eval_ci.py)
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. sys.exit(1) on threshold breach is HOW you fail a GitHub Actions step.
#    This is the entire mechanism — non-zero exit code = job fails = PR blocked.
# 2. Print clear output — GitHub Actions captures stdout as job logs.
#    Your future self will thank you when debugging why a CI run failed.
# 3. Keep the script self-contained (no notebook imports) — it must run
#    cleanly in a fresh CI environment with just `pip install` and `python`.

EVAL_SCRIPT = '''
#!/usr/bin/env python3
"""CI eval script — exits 1 if any metric is below threshold."""
import json
import sys
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

THRESHOLDS = {
    "avg_correctness": 3.0,
    "avg_conciseness": 3.0,
    "avg_tone":        3.5,
}

class EvalScore(BaseModel):
    correctness: int = Field(description="0-5: factual accuracy")
    conciseness: int = Field(description="0-5: avoid verbosity")
    tone:        int = Field(description="0-5: professional tone")
    reasoning:   str = Field(description="Brief justification")

JUDGE_SYSTEM = """
Score the response (0-5 per dimension).
5=excellent, 3=acceptable, 1=poor, 0=unacceptable.
"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge = llm.with_structured_output(EvalScore)

def score_case(case):
    prompt = f"QUESTION: {case[\"question\"]}\\nGROUND TRUTH: {case[\"ground_truth\"]}\\nRESPONSE: {case[\"agent_response\"]}"
    return judge.invoke([SystemMessage(content=JUDGE_SYSTEM), HumanMessage(content=prompt)])

def main():
    cases = json.loads(Path("eval_cases.json").read_text())
    scores = [score_case(c) for c in cases]
    
    means = {
        "avg_correctness": sum(s.correctness for s in scores) / len(scores),
        "avg_conciseness": sum(s.conciseness for s in scores) / len(scores),
        "avg_tone":        sum(s.tone for s in scores) / len(scores),
    }
    
    print("\\n=== EVAL RESULTS ===")
    failed = False
    for metric, score in means.items():
        threshold = THRESHOLDS[metric]
        status = "PASS" if score >= threshold else "FAIL"
        if status == "FAIL":
            failed = True
        print(f"  {metric:<20} {score:.2f}  (threshold={threshold})  [{status}]")
    
    if failed:
        print("\\n❌ Eval FAILED — one or more metrics below threshold")
        sys.exit(1)
    else:
        print("\\n✅ All metrics passed")
        sys.exit(0)

if __name__ == "__main__":
    main()
'''

# Write the script to disk
from pathlib import Path
script_path = Path("eval_ci.py")
script_path.write_text(EVAL_SCRIPT.strip())
print(f"✅ eval_ci.py written to {script_path.resolve()}")
print("\nTo test locally:")
print("  python eval_ci.py")
print("  echo $?  # 0 = all pass, 1 = threshold breached")

### 🧪 Try It Yourself — 10.2: Write the GitHub Actions workflow

Create `.github/workflows/eval.yml` that:
1. Triggers on `pull_request` to `main`
2. Checks out code
3. Sets up Python 3.11
4. Runs `pip install -r requirements.txt`
5. Runs `python eval_ci.py`
6. Uses GitHub Actions Secrets for `OPENAI_API_KEY` and `LANGCHAIN_API_KEY`

Then simulate a bad agent response in `eval_cases.json`, push to a branch, open a PR, and watch it fail.

In [ ]:
# 🧪 YOUR CODE HERE — Day 10.2: Generate the GitHub Actions workflow YAML
# Write the YAML to .github/workflows/eval.yml
# Then push to GitHub and open a test PR to verify it runs

In [ ]:
# ✅ REFERENCE SOLUTION — Day 10.2: GitHub Actions eval workflow
# ─────────────────────────────────────────────────────────────────────────────
# KEY INSIGHTS:
# 1. secrets.OPENAI_API_KEY is HOW you pass API keys securely to CI.
#    Never hardcode keys. Add them in GitHub repo Settings → Secrets.
# 2. The `if: failure()` step lets you post a comment on the PR showing which
#    metrics failed — much better UX than just a red X.
# 3. paths: ["eval_cases.json", "eval_ci.py", "src/**"] limits the trigger —
#    don't waste CI budget on README changes.

from pathlib import Path

WORKFLOW_YAML = """\
name: LLM Eval Gate

on:
  pull_request:
    branches: [ main ]
    paths:
      - 'eval_cases.json'
      - 'eval_ci.py'
      - 'src/**'

jobs:
  eval:
    runs-on: ubuntu-latest
    timeout-minutes: 10

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run eval pipeline
        env:
          OPENAI_API_KEY:      ${{ secrets.OPENAI_API_KEY }}
          LANGCHAIN_API_KEY:   ${{ secrets.LANGCHAIN_API_KEY }}
          LANGCHAIN_TRACING_V2: 'true'
          LANGCHAIN_PROJECT:   'ci-eval'
        run: python eval_ci.py

      - name: Post failure comment
        if: failure()
        uses: actions/github-script@v7
        with:
          script: |
            github.rest.issues.createComment({
              issue_number: context.issue.number,
              owner: context.repo.owner,
              repo: context.repo.repo,
              body: '❌ **LLM Eval Failed** — one or more metrics below threshold. Check the Actions log for details.'
            })
"""

workflow_path = Path(".github/workflows/eval.yml")
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(WORKFLOW_YAML)
print(f"✅ Workflow written to {workflow_path}")
print("\nNext steps:")
print("  1. git add .github/workflows/eval.yml eval_ci.py eval_cases.json")
print("  2. git commit -m 'feat: add LLM eval CI gate'")
print("  3. git push origin main")
print("  4. Create a branch, degrade one response in eval_cases.json, open a PR")
print("  5. Watch the Actions tab — the PR should fail")

### ✅ Day 10 Checkpoint

- [ ] Your eval script exits 1 on threshold breach — you've verified this locally  
- [ ] `.github/workflows/eval.yml` exists and triggers on PR  
- [ ] OPENAI_API_KEY is stored as a GitHub Secret (not in any file)  
- [ ] You can simulate a regression (bad response in eval_cases.json) and watch CI catch it  
- [ ] You know the difference between a regression test and a standard eval run  

**📝 Write LinkedIn Post #4 tonight** — open `02_linkedin_posts.md` → Post 4 (Day 10)

---

## 🏁 Days 6–10 Sprint Complete

**What you've built:**

| Deliverable | Day |
|---|---|
| RAGAS eval on a RAG pipeline — 4 metric scores, hallucination detected | Day 6 |
| Custom LLM-as-judge evaluator — correctness, conciseness, tone | Day 7 |
| LangSmith tracing — automatic, zero code change, metadata tagged | Day 8 |
| Guardrailed LangGraph agent — input filter + output sanitiser | Day 9 |
| GitHub Actions eval gate — PR blocked on metric regression | Day 10 |

**Move to `Code_Along_Day_11_14.ipynb` for asyncio + Python Concurrency.**